# Silver Layer — Boletos
Reads raw boletos from the **Bronze** Delta table, applies cleaning and enrichment transformations, and writes the result to the **Silver** Delta table.

**Medallion flow:** `Bronze (raw CSV)` → **`Silver (cleaned & enriched)`** → Gold (aggregated)

## 1. Imports & SparkSession

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType, DateType, StructType, StructField, StringType, BooleanType, IntegerType
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('Silver Boletos')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.1.1


## 2. Read from Bronze
Load the raw Delta table written by the ingestion notebook.

In [16]:
BRONZE_PATH = '../../output_data/bronze/boletos'

schema = StructType([
    StructField('id_boleto',       StringType(), nullable=True),
    StructField('id_pagador',      StringType(), nullable=True),
    StructField('id_beneficiario', StringType(), nullable=True),
    StructField('dt_emissao',      StringType(), nullable=True),
    StructField('dt_vencimento',   StringType(), nullable=True),
    StructField('dt_pagamento',    StringType(), nullable=True),
    StructField('vlr_nominal',     DoubleType(), nullable=True),
    StructField('vlr_baixa',       DoubleType(), nullable=True),
    StructField('tipo_baixa',      StringType(), nullable=True),
    StructField('tipo_especie',    StringType(), nullable=True),
    StructField('partition_date',   DateType(), nullable=False),
    StructField('ingestion_timestamp', TimestampType(), nullable=False)
])

df_bronze = spark.read.format('delta').load(BRONZE_PATH)

print(f'Rows in bronze: {df_bronze.count():,}')
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

Rows in bronze: 7,118
root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: string (nullable = true)
 |-- dt_vencimento: string (nullable = true)
 |-- dt_pagamento: string (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)

+----------------------------------------------------------------+----------------------------------------------------------------+----------------------------------------------------------------+----------+-------------+------------+-----------+---------+---------------------------------------------+---------------------------------+--------------------------+--------------+
|id_boleto                                                      

## 3. Data Quality Checks
Before transforming, understand what needs to be fixed.

In [17]:
total = df_bronze.count()

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in df_bronze.columns
])

null_pct = null_counts.select([
    F.round(F.col(c) / total * 100, 2).alias(c)
    for c in df_bronze.columns
])

print('Null counts:')
null_counts.show(truncate=False)
print('Null % per column:')
null_pct.show(truncate=False)

Null counts:
+---------+----------+---------------+----------+-------------+------------+-----------+---------+----------+------------+-------------------+--------------+
|id_boleto|id_pagador|id_beneficiario|dt_emissao|dt_vencimento|dt_pagamento|vlr_nominal|vlr_baixa|tipo_baixa|tipo_especie|ingestion_timestamp|partition_date|
+---------+----------+---------------+----------+-------------+------------+-----------+---------+----------+------------+-------------------+--------------+
|0        |0         |0              |0         |0            |70          |0          |820      |70        |0           |0                  |0             |
+---------+----------+---------------+----------+-------------+------------+-----------+---------+----------+------------+-------------------+--------------+

Null % per column:
+---------+----------+---------------+----------+-------------+------------+-----------+---------+----------+------------+-------------------+--------------+
|id_boleto|id_pagad

In [18]:
total_rows   = df_bronze.count()
unique_ids   = df_bronze.select('id_boleto').distinct().count()
duplicates   = total_rows - unique_ids

print(f'Total rows    : {total_rows:,}')
print(f'Unique boletos: {unique_ids:,}')
print(f'Duplicates    : {duplicates:,}')

Total rows    : 7,118
Unique boletos: 7,118
Duplicates    : 0


## 4. Transformations
Apply all cleaning and enrichment steps to produce the silver DataFrame.

In [19]:
# 4.1 — Cast string dates to DateType
df = (
    df_bronze
    .withColumn('dt_emissao',    F.to_date('dt_emissao',    'yyyy-MM-dd'))
    .withColumn('dt_vencimento', F.to_date('dt_vencimento', 'yyyy-MM-dd'))
    .withColumn('dt_pagamento',  F.to_date('dt_pagamento',  'yyyy-MM-dd'))
)
print('Date columns cast to DateType.')

Date columns cast to DateType.


In [20]:
# 4.2 — Fill missing vlr_baixa with 0.0 (boleto was never paid)
df = df.withColumn('vlr_baixa', F.coalesce(F.col('vlr_baixa'), F.lit(0.0)))

print('Missing vlr_baixa filled with 0.0')
df.select('vlr_baixa').summary('count', 'min', 'max').show()

Missing vlr_baixa filled with 0.0
+-------+----------+
|summary| vlr_baixa|
+-------+----------+
|  count|      7118|
|    min|       0.0|
|    max|1019818.73|
+-------+----------+



In [21]:
df.printSchema()

root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = false)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)



In [22]:
df.printSchema()

root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = false)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)



In [23]:

df_silver = (
    df
    .withColumn('boleto_pago', F.col('dt_pagamento').isNotNull())

    .withColumn('dias_ate_vencimento',
        F.datediff('dt_vencimento', 'dt_emissao'))

    .withColumn('dias_atraso',
        F.when(
            F.col('boleto_pago'),
            F.datediff('dt_pagamento', 'dt_vencimento')
        ).otherwise(F.lit(None)))

    .withColumn('boleto_pago_em_atraso',
        F.when(F.col('boleto_pago'), F.col('dias_atraso') > 0)
        .otherwise(F.lit(None)))

    # Diferença aplicada (nominal - paid amount), apenas para boletos pagos
    .withColumn('vlr_diferenca',
        F.when(
            F.col('boleto_pago') & (F.col('vlr_baixa') > 0),
            F.round(F.col('vlr_nominal') - F.col('vlr_baixa'), 2)
        ).otherwise(F.lit(None)))

    .withColumn('mes_emissao', F.date_format('dt_emissao', 'yyyy-MM'))

    .withColumn("categoria_baixa",
        F.when(F.col("tipo_baixa").contains("interbancaria"), "Interbancaria")
         .when(F.col("tipo_baixa").contains("solicitacao do cedente"), "Solicitacao Cedente")
         .when(F.col("tipo_baixa").contains("solicitacao da instituicao"), "Solicitacao Instituicao")
         .otherwise("Outros"))

     # --- Faixa de valor ---
    .withColumn("faixa_valor",
        F.when(F.col("vlr_nominal") < 500, "Ate 500")
         .when(F.col("vlr_nominal") < 2000, "500-2k")
         .when(F.col("vlr_nominal") < 10000, "2k-10k")
         .when(F.col("vlr_nominal") < 50000, "10k-50k")
         .otherwise("50k+"))

    .withColumn('tipo_baixa',
        F.when(F.col('tipo_baixa').isNull(),'Sem baixa')
        .otherwise(F.col('tipo_baixa')))

    # --- Extracão temporal ---
    .withColumn("ano_mes_emissao", F.date_format("dt_emissao", "yyyy-MM"))
    .withColumn("ano_mes_vencimento", F.date_format("dt_vencimento", "yyyy-MM"))

    .withColumn('ingestion_timestamp', F.current_timestamp())

    .withColumn('partition_date', F.current_date())
)

df_silver.printSchema()

root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = false)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- partition_date: date (nullable = false)
 |-- boleto_pago: boolean (nullable = false)
 |-- dias_ate_vencimento: integer (nullable = true)
 |-- dias_atraso: integer (nullable = true)
 |-- boleto_pago_em_atraso: boolean (nullable = true)
 |-- vlr_diferenca: double (nullable = true)
 |-- mes_emissao: string (nullable = true)
 |-- categoria_baixa: string (nullable = false)
 |-- faixa_valor: string (nullable = false)
 |-- ano_mes_emissao: string (nullable = true)
 |-- ano_mes_vencimento: string (nullable = tru

## 5. Validation
Quick sanity checks before writing.

In [24]:
paid     = df_silver.filter(F.col('boleto_pago')).count()
unpaid   = df_silver.filter(~F.col('boleto_pago')).count()
late     = df_silver.filter(F.col('boleto_pago_em_atraso') == True).count()
early    = df_silver.filter(F.col('boleto_pago_em_atraso') == False).count()

print(f'Total rows      : {df_silver.count():,}')
print(f'Paid            : {paid:,}')
print(f'Unpaid          : {unpaid:,}')
print(f'Paid late       : {late:,}')
print(f'Paid on time    : {early:,}')

# Check no negative vlr_nominal
neg_nominal = df_silver.filter(F.col('vlr_nominal') < 0).count()
print(f'Negative vlr_nominal: {neg_nominal:,}')

Total rows      : 7,118
Paid            : 7,048
Unpaid          : 70
Paid late       : 2,087
Paid on time    : 4,961
Negative vlr_nominal: 0


## 6. Write to Silver
Persist the cleaned and enriched DataFrame as a Delta table.

In [25]:
SILVER_PATH = '../../output_data/silver/boletos'

(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .partitionBy('partition_date')
    .option('overwriteSchema', 'true')
    .save(SILVER_PATH)
)

print(f'Silver layer saved to {SILVER_PATH}')

Silver layer saved to ../../output_data/silver/boletos


In [26]:
df_check = spark.read.format('delta').load(SILVER_PATH)
print(f'Rows in silver: {df_check.count():,}')
df_check.printSchema()

Rows in silver: 7,118
root
 |-- id_boleto: string (nullable = true)
 |-- id_pagador: string (nullable = true)
 |-- id_beneficiario: string (nullable = true)
 |-- dt_emissao: date (nullable = true)
 |-- dt_vencimento: date (nullable = true)
 |-- dt_pagamento: date (nullable = true)
 |-- vlr_nominal: double (nullable = true)
 |-- vlr_baixa: double (nullable = true)
 |-- tipo_baixa: string (nullable = true)
 |-- tipo_especie: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)
 |-- boleto_pago: boolean (nullable = true)
 |-- dias_ate_vencimento: integer (nullable = true)
 |-- dias_atraso: integer (nullable = true)
 |-- boleto_pago_em_atraso: boolean (nullable = true)
 |-- vlr_diferenca: double (nullable = true)
 |-- mes_emissao: string (nullable = true)
 |-- categoria_baixa: string (nullable = true)
 |-- faixa_valor: string (nullable = true)
 |-- ano_mes_emissao: string (nullable = true)
 |-- ano_mes_vencimento: string

In [27]:
df_silver.select('tipo_baixa').distinct().show()

+--------------------+
|          tipo_baixa|
+--------------------+
|8 - Baixa integra...|
|           Sem baixa|
|9 - Baixa integra...|
|5 - Baixa integra...|
|6 - Baixa integra...|
|0 - Baixa integra...|
|7 - Baixa integra...|
|1 - Baixa integra...|
+--------------------+

